# N060 · 双堆中位数与延迟删除

**目标：** 维护两个半区的秩关系并剔除失效堆顶。

**先修：** N057, N058, N028。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 平衡；逻辑大小；延迟计数字典；过期元素；重复值。

## 从问题到算法

中位数只需要分隔较小一半与较大一半：小半用最大堆，大半用最小堆。滑动窗口删除的元素可能不在堆顶，不能用线性查找假装O(log k)。延迟删除按值计数，并分别维护逻辑有效大小；访问或移动堆顶前清理失效项。物理堆含过期元素，本实现最坏空间可达O(n)，不是无条件O(k)。

## 最小实现

**本章接口：** `MedianFinder`、`sliding_window_median(nums, k)`

In [1]:
import heapq
from collections import Counter

class MedianFinder:
    def __init__(self): self.low=[]; self.high=[]
    def addNum(self,x):
        heapq.heappush(self.low,-x)
        heapq.heappush(self.high,-heapq.heappop(self.low))
        if len(self.high)>len(self.low): heapq.heappush(self.low,-heapq.heappop(self.high))
    def findMedian(self):
        if not self.low: raise ValueError('no observations')
        return -self.low[0] if len(self.low)>len(self.high) else (-self.low[0]+self.high[0])/2

class _WindowMedian:
    def __init__(self):
        self.low=[]; self.high=[]; self.delayed=Counter(); self.nl=self.nh=0
    def prune(self,heap,sign):
        while heap and self.delayed[sign*heap[0]]:
            x=sign*heapq.heappop(heap); self.delayed[x]-=1
            if not self.delayed[x]: del self.delayed[x]
    def balance(self):
        if self.nl>self.nh+1:
            heapq.heappush(self.high,-heapq.heappop(self.low)); self.nl-=1; self.nh+=1
            self.prune(self.low,-1)
        elif self.nl<self.nh:
            heapq.heappush(self.low,-heapq.heappop(self.high)); self.nl+=1; self.nh-=1
            self.prune(self.high,1)
    def add(self,x):
        if not self.low or x<=-self.low[0]: heapq.heappush(self.low,-x); self.nl+=1
        else: heapq.heappush(self.high,x); self.nh+=1
        self.balance()
    def remove(self,x):
        self.delayed[x]+=1
        if x<=-self.low[0]:
            self.nl-=1
            if x==-self.low[0]: self.prune(self.low,-1)
        else:
            self.nh-=1
            if self.high and x==self.high[0]: self.prune(self.high,1)
        self.balance()
    def median(self):
        return -self.low[0] if self.nl>self.nh else (-self.low[0]+self.high[0])/2

def sliding_window_median(nums,k):
    if not 1<=k<=len(nums): raise ValueError('invalid window size')
    state=_WindowMedian(); out=[]
    for x in nums[:k]: state.add(x)
    out.append(state.median())
    for i in range(k,len(nums)):
        state.add(nums[i]); state.remove(nums[i-k]); out.append(state.median())
    return out

## 正确性、前提与复杂度

两半有效元素保持low≤high且大小差为0或1，中位数由边界直接决定。失效元素只在到达顶端时移除，逻辑大小先减，不能拿物理len代替逻辑计数。

**代价：** 在线插入O(log n)时间/O(n)空间。延迟滑窗每项入出堆至多常数次，本实现保守界O(n log n)时间/O(n)空间；定期重建可进一步约束空间。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

m=MedianFinder(); rows=[]
for x in [1,2,3,9,0,5]:
    m.addNum(x); rows.append((x,sorted(-v for v in m.low),sorted(m.high),m.findMedian()))
show_table(['新值','较小一半','较大一半','中位数'],rows)

新值,较小一半,较大一半,中位数
1,[1],[],1
2,[1],[2],1.5
3,"[1, 2]",[3],2
9,"[1, 2]","[3, 9]",2.5
0,"[0, 1, 2]","[3, 9]",2
5,"[0, 1, 2]","[3, 5, 9]",2.5


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from statistics import median
from random import Random
m=MedianFinder(); m.addNum(1); m.addNum(2); assert m.findMedian()==1.5
m.addNum(3); assert m.findMedian()==2
rng=Random(60)
for n in range(1,35):
    for a in [[3]*n,list(range(n)),list(range(n,0,-1)),[rng.randrange(-3,4) for _ in range(n)]]:
        for k in range(1,n+1): assert sliding_window_median(a,k)==[median(a[i:i+k]) for i in range(n-k+1)]
print('N060: 所有本章断言通过')

N060: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 解释物理堆长不等于有效大小。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 处理连续相等值的过期元素。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 295. Find Median from Data Stream（canonical）
- 480. Sliding Window Median（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。